In [ ]:
from pathlib import Path
import flammkuchen as fl
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# calculate motor tuning from dF/F traces, px-wise
def get_tuning_map(img, mot_regs, n_dirs=2):
    traces = img.reshape(img.shape[0], -1)

    n_t = sens_regs.shape[0]
    reg = sens_regs.values.T @ traces[:n_t, :]
    reg = reg.reshape(reg.shape[0], img.shape[-2], img.shape[-1])
    
    # tuning vector
    bin_centers, bins = quantize_directions([0], n_dirs)
    vectors = np.stack([np.cos(bin_centers), np.sin(bin_centers)], 0)
    reg_vectors = np.reshape(
        vectors @ np.reshape(reg[:, :, :], (n_dirs, -1)),
        (2,) + reg.shape[1:],
    )
    angle = np.arctan2(reg_vectors[1], reg_vectors[0])
    amp = np.sqrt(np.sum(reg_vectors ** 2, 0))

    return amp, angle

In [ ]:
master = Path(r"Z:\Hagar\e0075\v04_4x4")
fish_list = list(master.glob("*_f*"))
fish = fish_list[33]
print(fish)

In [ ]:
corrmap_all = fl.load(fish / "plane_corrmap_motor.h5")['plane_corr']

In [ ]:
ind_plane = 0
corrmap = corrmap_all[ind_plane]

In [ ]:
num_row = 1
num_col = 3
fig, axs = plt.subplots(num_row, num_col, figsize=(10, 4), sharey=True, sharex=True)

title_list = ['Left', 'Right', 'Forward']

vmax = 0.25
fig.suptitle("Plane " + str (ind_plane) + ', corr thresh=' + str(vmax))

for i in range(0, num_col):

    axs[i].axis('off')
    axs[i].set_title(title_list[i])  
    tmp_plane = np.rot90(corrmap[i], 2)
    axs[i].imshow(tmp_plane, cmap='coolwarm', vmin=-vmax, vmax=vmax)

file_name = "plane" + str(ind_plane) + "_regs_map_motor_corrval_025.jpg"
fig.savefig(fish / file_name, dpi=300)